# Notebook 02 – Data Cleaning

## Objective
Prepare the dataset for analysis by identifying and resolving data quality issues while preserving data integrity.

---

### Checklist

- [x] Import libraries
- [x] Load the dataset
- [x] Create a working copy
- [x] Inspect dataset structure
- [x] Handle duplicate records
- [x] Handle missing values
- [x] Validate data types
- [x] Standardize categorical values (if needed)
- [X] Detect potential data quality issues
- [X] Validate the cleaned dataset
- [X] Save the cleaned dataset

## 1. Import Libraries

Import the required libraries to support data manipulation, data cleaning, and handling missing values throughout this notebook.

In [ ]:
import pandas as pd

## 2. Load Dataset

Load the original dataset to begin the data cleaning process. Since no modifications were made during the Data Understanding stage, the raw dataset will be used as the starting point.

In [ ]:
df = pd.read_excel('../data/online_retail_II.xlsx')

## 3. Create a Working Copy

Create a separate working copy of the dataset to ensure the original data remains unchanged throughout the cleaning process. This helps preserve the raw data and provides a safe environment for applying transformations.

In [ ]:
df_clean = df.copy()

## 4. Inspect Dataset Structure

Review the structure of the working dataset to verify the available columns, data types, and completeness before starting the cleaning process.

In [ ]:
df_clean.info()

## 5. Handle Duplicate Records

Identify duplicate rows in the dataset to determine whether they represent data quality issues before deciding whether any records should be removed.

### 5.1 Check for Duplicate Records

In [ ]:
df_clean.duplicated().sum()

### 5.2 Inspect Duplicate Samples

In [ ]:
duplicates = df_clean[df_clean.duplicated(keep=False)]
duplicates.sort_values(by=df.columns.to_list()).head(20)

### 5.3 Remove Duplicate Records

In [ ]:
rows_before = df_clean.shape[0]
duplicate_count = df_clean.duplicated().sum()

df_clean.drop_duplicates(inplace=True)

rows_after = df_clean.shape[0]
removed_rows = rows_before - rows_after

duplicate_count == removed_rows
rows_before, rows_after, removed_rows

### 5.4 Verify Removal

In [ ]:
pd.Series({'No Duplicated Record' : df_clean.duplicated().sum() == 0})

### 5.5 Interpretation

The dataset originally contained **525,461** rows. After removing exact duplicate records, the dataset was reduced to **518,596** rows, resulting in **6,865** records being removed.

Removing these exact duplicates helps prevent double-counting transactions and improves the overall quality of the dataset. This ensures that subsequent analyses and visualizations are based on unique records, leading to more accurate and reliable insights.

## 6. Handle Missing Values

Identify missing values, evaluate their impact, and apply appropriate cleaning strategies while preserving as much useful information as possible.

### 6.1 Check For Missing Values 

In [ ]:
df_clean.isna().any().any()

### 6.2 Inspect Missing Values

In [ ]:
missing_count = df_clean.isna().sum()

missing_count_filtered = missing_count[missing_count > 0]

missing_percentage = ((missing_count_filtered / df_clean.shape[0]) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count' : missing_count_filtered,
    'Missing Percentage' : missing_percentage
})

missing_summary = missing_summary.sort_values(by='Missing Count', ascending=False)

missing_summary.index.name = 'Column'

missing_summary

### 6.3 Decision

- **Customer ID:** Missing values will be retained because removing these records would discard a significant portion of the dataset (20.79%). Although customer-level analyses require this field, the remaining information is still valuable for sales-related analyses. Customer-based analyses can later be performed using only records with valid customer IDs.

- **Description:** Rows with missing product descriptions will be removed. These records represent only **0.56%** of the dataset, so removing them is unlikely to affect the overall analysis. While missing descriptions could potentially be recovered using the corresponding **StockCode**, this enhancement is beyond the scope of the current notebook and may be considered in future work.

### 6.4 Handle Missing Values

In [ ]:
rows_before = df_clean.shape[0]

missing_description_count = df_clean['Description'].isna().sum()

df_clean.dropna(subset=['Description'], inplace=True)

### 6.5 Verify Changes

In [ ]:
rows_after = df_clean.shape[0]

deleted_rows = rows_before - rows_after

remaining_missing = df_clean['Description'].isna().sum()

pd.Series({
    'Deleted rows verified': deleted_rows == missing_description_count,
    'No missing descriptions': remaining_missing == 0
})

### 6.6 Interpretation

The dataset size decreased from **518,596** to **515,668** rows after removing **2,928** records with missing values in the **Description** column. Since these records represented only **0.56%** of the dataset, their removal had a negligible impact on the overall data while improving dataset consistency by ensuring that every remaining transaction has a valid product description.

Missing values in the **Customer ID** column were intentionally retained because removing them would have eliminated **20.79%** of the dataset. Although customer-level analyses require this field, the remaining transaction information is still valuable for product- and sales-level analyses.

## 7. Validate Data Types

### 7.1 Check

In [ ]:
df_clean.dtypes

### 7.2 Inspect

In [ ]:
df_clean['Customer ID'].head(10)

df_clean['Customer ID'].dropna().mod(1).eq(0).all()

### 7.3 Decision

The **Customer ID** column is currently stored as `float64` due to the presence of missing values (NaN). Since this column represents an identifier rather than a numerical variable, it should be treated as a categorical/integer-like field rather than a continuous numeric feature.

To maintain data consistency while still preserving missing values, the column will be converted to pandas' nullable integer type (`Int64`). This allows integer values to remain intact while supporting missing values (`NaN`).

### 7.4 Handle

In [ ]:
df_clean['Customer ID'] = df_clean['Customer ID'].astype('Int64')

### 7.5 Verify

In [ ]:
pd.Series({
    'Customer ID type is Int64' : df_clean['Customer ID'].dtype == 'Int64'
    })

### 7.6 Interpretation

The **Customer ID** column was successfully converted from `float64` to pandas' nullable integer type (`Int64`). This change ensures that the column correctly reflects its role as an identifier rather than a continuous numerical variable, while still preserving missing values.

No data loss occurred during the conversion, and all values remain integer-like, confirming the correctness of the transformation.

## 8. Standardize Categorical Values

### 8.1 Check & Inspect Categorical Values  

In [ ]:
cat_cols = df_clean.select_dtypes(include=['object', 'string'])

for col in cat_cols:
    print(f"\n{col}")
    print(df_clean[col].head(5))

In [ ]:
original_counts = df_clean['Country'].value_counts()
stripped_counts = df_clean['Country'].str.strip().value_counts()

(original_counts == stripped_counts).all()

In [ ]:
original_sc = df_clean['StockCode'].astype(str).value_counts()
stripped_sc = df_clean['StockCode'].astype(str).str.strip().value_counts()

original_sc.sort_index().equals(stripped_sc.sort_index())

In [ ]:
df_clean['StockCode'].astype(str).str.contains(' ').value_counts()

In [ ]:
df_clean[df_clean['StockCode'].astype(str).str.contains(' ')]['StockCode'].head(10)

### 8.2 Decision 

We decide to apply light standardization to StockCode.

- Leading and trailing spaces were found to affect category consistency.
- However, StockCode remains a structured identifier and should not be transformed into a numeric variable.
- Therefore, only whitespace cleaning (str.strip) will be applied, without altering the underlying code structure.

### 8.3 Handle

In [ ]:
df_clean['StockCode'] = df_clean['StockCode'].astype(str).str.strip()

### 8.4 Verify  

In [ ]:
(
    df_clean['StockCode']
    .astype(str)
    .str.strip()
    .equals(df_clean['StockCode'].astype(str))
)

### 8.6 Interpretation 

Minor inconsistencies were detected in the StockCode column in the form of leading and trailing whitespace. These inconsistencies affected categorical grouping, as confirmed during inspection.

A minimal standardization was applied by removing whitespace while preserving the original structure and meaning of the codes. No semantic transformation was performed to ensure data integrity.

This ensures consistent grouping of product identifiers without altering their underlying representation.

## 9. Detect Potential Data Quality Issues

### 9.1 Check
- Negative Quantity
- Negative Price
- Zero Price
- Cancelled Invoice

In [ ]:
pd.Series({
    'Negative Quantity': (df_clean['Quantity'] < 0).sum(),
    'Negative Price': (df_clean['Price'] < 0).sum(),
    'Zero Price': (df_clean['Price'] == 0).sum(),
    'Cancelled Invoices': df_clean['Invoice'].astype(str).str.startswith('C').sum()
})

### 9.2 Inspect

In [ ]:
negative_quantity = df_clean['Quantity'] < 0
cancelled_invoice = df_clean['Invoice'].astype(str).str.startswith('C')

pd.Series({
    'Negative Quantity': negative_quantity.sum(),
    'Cancelled Invoice': cancelled_invoice.sum(),
    'Negative Quantity & Cancelled Invoice': (negative_quantity & cancelled_invoice).sum()
})

In [ ]:
negative_without_cancel = df_clean[
    (df_clean['Quantity'] < 0) &
    (~df_clean['Invoice'].astype(str).str.startswith('C'))
]

negative_without_cancel.head(10)

To better understand cancelled transactions, we verify whether they retain customer identifiers.

In [ ]:
cancelled = df_clean['Invoice'].astype(str).str.startswith('C')

df_clean.loc[cancelled, 'Customer ID'].isna().value_counts()

### 9.3 Interpretation

During inspection, negative quantities were found to represent both cancelled sales and inventory adjustment activities. A sample investigation showed that cancelled records may correspond to separate positive sales records, but identifying these pairs reliably would require a dedicated record-matching process. Since the objective of this project is data cleaning for sales analysis rather than transaction reconciliation, this matching process is considered outside the current project scope.

### 9.4 Decision

Negative quantities were found to represent two different types of valid business transactions: inventory adjustments (e.g., damaged, lost, stock corrections) and cancelled customer transactions. These records are not data entry errors; however, they do not represent completed customer sales.

Since the objective of this project is to prepare a dataset for sales analysis rather than inventory management or transaction reconciliation, both inventory adjustment records and cancelled invoices will be excluded from the analytical dataset.

It is acknowledged that removing cancelled invoices without matching them to their corresponding original sales may slightly overestimate gross sales. A reliable reconciliation between original and cancelled transactions requires a dedicated record-matching procedure, which is beyond the scope of this project.


### 9.4 Handle

In [ ]:
rows_before = df_clean.shape[0]

inventory_adjustments = (
    (df_clean['Quantity'] < 0) &
    (~df_clean['Invoice'].astype(str).str.startswith('C'))
)

cancelled_invoices = (
    df_clean['Invoice'].astype(str).str.startswith('C')
)

accounting_adjustments = (
    (df_clean['Price'] < 0) &
    (df_clean['Description'] == 'Adjust bad debt')
)

df_clean = df_clean[
    ~(inventory_adjustments |
      cancelled_invoices |
      accounting_adjustments)
].copy()

rows_after = df_clean.shape[0]

deleted_rows = rows_before - rows_after

### 9.5 Verify

In [ ]:
pd.Series({
    'No negative inventory adjustments':
        (
            (df_clean['Quantity'] < 0) &
            (~df_clean['Invoice'].astype(str).str.startswith('C'))
        ).sum() == 0,

    'No cancelled invoices':
        df_clean['Invoice'].astype(str).str.startswith('C').sum() == 0,

    'No negative prices':
        (df_clean['Price'] < 0).sum() == 0,

    'Rows removed':
        deleted_rows > 0
})

## 10. Validate the cleaned dataset

### 10.1 Check

In [ ]:
pd.Series({
    'No missing descriptions':
        df_clean['Description'].isna().sum() == 0,

    'Customer ID is Int64':
        df_clean['Customer ID'].dtype == 'Int64',

    'No cancelled invoices':
        (~df_clean['Invoice'].astype(str).str.startswith('C')).all(),

    'No negative quantities':
        (df_clean['Quantity'] >= 0).all(),

    'No negative prices':
        (df_clean['Price'] >= 0).all()
})

In [ ]:
df_clean[df_clean['Price'] < 0]

### 10.2 Inspect

In [ ]:
df_clean.info()

In [ ]:
df_clean.head()

### 10.3 Interpretation

The validation confirms that all planned cleaning operations have been successfully applied. The dataset is internally consistent, free from the targeted data quality issues, and ready for exploratory data analysis.

### 10.4 Decision

The cleaned dataset satisfies the project requirements and is approved for the next stage of the analysis.

## 11. Save the cleaned dataset

### 11.1 Decision

The cleaned dataset has passed all validation checks and is ready to be saved for future analysis.

### 11.2 Handle

In [112]:
df_clean.to_csv(
    '../data/cleaned/online_retail_clean.csv',
    index=False
)

### 11.3 Verify

In [113]:
import os

os.path.exists('../data/cleaned/online_retail_clean.csv')

True

## Summary

- X rows removed
- Missing descriptions handled
- Customer IDs converted
- Cancelled invoices removed
- Dataset ready for EDA